# Session 2 — Extras

Dieses Notebook enthält optionale Inhalte für Teilnehmer, die die Kerninhalte aus den Notebooks 3–6 bereits abgeschlossen haben.

Die Themen hier sind **nicht Voraussetzung für weiterführende Workshops** — sie machen ETL-Pipelines aber robuster, effizienter und professioneller.

## Inhalte

1. Reguläre Ausdrücke (Regex) zur Textbereinigung
2. Parquet — effizientes Dateiformat
3. Erweiterte SQL-Abfragen (Window Functions, CTEs)
4. Interaktive Dashboards mit plotly
5. Logging in ETL-Pipelines
6. Große Dateien mit `chunksize` verarbeiten

---

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os

os.makedirs("output", exist_ok=True)
os.makedirs("daten", exist_ok=True)
print("Bereit")

---
# 1. Reguläre Ausdrücke (Regex)

Reguläre Ausdrücke (Regular Expressions, kurz: Regex) sind eine Sprache zur Beschreibung von Textmustern. Sie sind extrem mächtig für die Bereinigung von Rohdaten.

Python stellt das `re`-Modul bereit; pandas integriert Regex direkt in die `.str`-Methoden.

## Grundlegende Muster

| Muster | Bedeutung | Beispiel |
|--------|-----------|----------|
| `.` | Beliebiges Zeichen | `a.c` trifft `abc`, `a1c` |
| `\d` | Ziffer (0–9) | `\d\d` trifft `42` |
| `\w` | Buchstabe, Ziffer oder `_` | `\w+` trifft `hallo_123` |
| `\s` | Whitespace (Leerzeichen, Tab) | `\s+` trifft mehrere Leerzeichen |
| `+` | 1 oder mehr | `\d+` trifft `1`, `42`, `1000` |
| `*` | 0 oder mehr | `\d*` trifft auch leeren String |
| `?` | 0 oder 1 | `colou?r` trifft `color` und `colour` |
| `^` | Anfang des Strings | `^Hallo` |
| `$` | Ende des Strings | `Euro$` |
| `[abc]` | Eines von a, b, c | `[aeiou]` = Vokal |
| `(...)` | Gruppe (zum Extrahieren) | `(\d+) Euro` |


In [ ]:
# re.search(): Gibt ein Match-Objekt zurück, wenn das Muster gefunden wird
text = "Bestellung B-2024-0042 wurde am 15.01.2024 geliefert."

# Datum finden
match = re.search(r"(\d{2}\.\d{2}\.\d{4})", text)
if match:
    print("Gefunden:", match.group(1))

# Bestellnummer finden
match_nr = re.search(r"B-\d{4}-\d{4}", text)
if match_nr:
    print("Bestellnummer:", match_nr.group())

In [ ]:
# re.findall(): Alle Treffer zurückgeben
text_mit_zahlen = "Q1: 45.000 Euro, Q2: 51.200 Euro, Q3: 48.900 Euro"

zahlen = re.findall(r"[\d\.]+", text_mit_zahlen)
print("Alle Zahlen:", zahlen)

# Nur ganze Zahlen (kein Punkt):
nur_ganze = re.findall(r"\d+", text_mit_zahlen.replace(".", ""))
print("Werte:", nur_ganze)

In [ ]:
# re.sub(): Ersetzen — mächtiger als .replace()
telefonnummern = [
    "040 / 12 34 56",
    "+49-40-123456",
    "(040)123456",
    "040.12.34.56",
]

# Alles außer Ziffern entfernen und normalisieren
def telefon_normalisieren(nummer):
    nur_ziffern = re.sub(r"[^\d]", "", nummer)  # alles außer Ziffern entfernen
    if nur_ziffern.startswith("49"):             # internationale Vorwahl
        nur_ziffern = "0" + nur_ziffern[2:]
    return nur_ziffern

for nr in telefonnummern:
    print(f"  '{nr}' -> '{telefon_normalisieren(nr)}'")

In [ ]:
# Regex in pandas — direkt auf Spalten anwenden
df_regex = pd.DataFrame({
    "beschreibung": [
        "Laptop Pro 15Zoll, Artikel-Nr: LPT-2024-001, Preis: 999 Euro",
        "USB-Maus kabellos, Artikel-Nr: MOU-2023-042, Preis: 29 Euro",
        "Monitor 27Zoll, Artikel-Nr: MON-2024-007, Preis: 349 Euro",
    ]
})

# Artikel-Nr und Preis extrahieren
df_regex["artikel_nr"] = df_regex["beschreibung"].str.extract(r"(\w{3}-\d{4}-\d{3})")
df_regex["preis"]      = df_regex["beschreibung"].str.extract(r"Preis: (\d+)").astype(int)
df_regex[["artikel_nr", "preis", "beschreibung"]]

### Übung 1: Regex

Extrahiere aus der Spalte `email` die Domain (alles nach dem `@`) als neue Spalte `domain`.

In [ ]:
df_email = pd.DataFrame({
    "name":  ["Anna", "Klaus", "Sara", "Tom"],
    "email": ["anna@firma.de", "k.mueller@beispiel.com", "s_wagner@test.org", "tom@firma.de"]
})

In [ ]:
# LÖSUNG
df_email["domain"] = df_email["email"].str.extract(r"@(.+)")
print("Häufigste Domain:", df_email["domain"].value_counts().idxmax())
df_email

---
# 2. Parquet — effizientes Dateiformat

CSV ist universell, aber ineffizient: alles wird als Text gespeichert, Datentypen gehen verloren.

**Parquet** ist ein spaltenorientiertes Binärformat, das:
- Datentypen erhält
- deutlich weniger Speicherplatz benötigt
- viel schneller zu lesen und schreiben ist
- der Standard in modernen Datenpipelines ist (z.B. Spark, dbt, Azure)



In [ ]:
try:
    import pyarrow  # Parquet-Engine

    # Beispiel-DataFrame mit verschiedenen Typen
    df_typen = pd.DataFrame({
        "id":       range(1000),
        "name":     [f"Kunde_{i}" for i in range(1000)],
        "umsatz":   np.random.uniform(100, 10000, 1000).round(2),
        "datum":    pd.date_range("2024-01-01", periods=1000, freq="h"),
        "aktiv":    np.random.choice([True, False], 1000),
    })

    # Als CSV speichern
    df_typen.to_csv("output/vergleich.csv", index=False)
    csv_groesse = os.path.getsize("output/vergleich.csv")

    # Als Parquet speichern
    df_typen.to_parquet("output/vergleich.parquet", index=False)
    parquet_groesse = os.path.getsize("output/vergleich.parquet")

    print(f"CSV:     {csv_groesse / 1024:.1f} KB")
    print(f"Parquet: {parquet_groesse / 1024:.1f} KB")
    print(f"Parquet ist {csv_groesse / parquet_groesse:.1f}x kleiner")

    # Einlesen — Datentypen bleiben erhalten!
    df_von_parquet = pd.read_parquet("output/vergleich.parquet")
    print("\nDatentypen nach dem Einlesen:")
    print(df_von_parquet.dtypes)

except ImportError:
    print("pyarrow nicht installiert — Parquet-Abschnitt wird übersprungen")

In [ ]:
try:
    # Nur bestimmte Spalten lesen (Parquet unterstützt column pruning)
    df_teilmenge = pd.read_parquet(
        "output/vergleich.parquet",
        columns=["id", "umsatz", "datum"]  # Nur diese Spalten laden
    )
    print(f"Nur 3 Spalten geladen: {df_teilmenge.shape}")
    df_teilmenge.head(3)
except Exception:
    print("Parquet-Datei nicht vorhanden — pyarrow installieren und vorherige Zelle ausführen.")

---
# 3. Erweiterte SQL-Abfragen

## Window Functions — Berechnungen über "Fenster" von Zeilen

Window Functions erlauben Berechnungen, die sich auf eine Gruppe beziehen, aber trotzdem jede Zeile einzeln behalten — z.B. kumulativer Umsatz, Rang innerhalb einer Gruppe.

In [ ]:
# Datenbank mit Beispieldaten
con = sqlite3.connect(":memory:")  # In-Memory-Datenbank (keine Datei)

pd.DataFrame([
    {"monat": 1, "region": "nord", "umsatz": 12000},
    {"monat": 1, "region": "süd",  "umsatz": 9500},
    {"monat": 1, "region": "ost",  "umsatz": 7800},
    {"monat": 2, "region": "nord", "umsatz": 15000},
    {"monat": 2, "region": "süd",  "umsatz": 11000},
    {"monat": 2, "region": "ost",  "umsatz": 9200},
    {"monat": 3, "region": "nord", "umsatz": 13500},
    {"monat": 3, "region": "süd",  "umsatz": 10200},
    {"monat": 3, "region": "ost",  "umsatz": 8800},
]).to_sql("verkauf", con, index=False)

print("Datenbank bereit.")

In [ ]:
# Window Function: Rang pro Monat (welche Region war pro Monat am stärksten?)
sql_rang = """
    SELECT
        monat,
        region,
        umsatz,
        RANK() OVER (PARTITION BY monat ORDER BY umsatz DESC) AS rang_im_monat
    FROM verkauf
    ORDER BY monat, rang_im_monat
"""
print("Rang pro Monat:")
display(pd.read_sql(sql_rang, con))

In [ ]:
# Window Function: Kumulativer Umsatz pro Region über die Monate
sql_kumulativ = """
    SELECT
        monat,
        region,
        umsatz,
        SUM(umsatz) OVER (
            PARTITION BY region
            ORDER BY monat
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS umsatz_kumulativ
    FROM verkauf
    ORDER BY region, monat
"""
print("Kumulativer Umsatz pro Region:")
display(pd.read_sql(sql_kumulativ, con))

In [ ]:
# CTE (Common Table Expression) — Zwischenergebnisse benennen
# Das macht lange Abfragen viel lesbarer.

sql_cte = """
    -- CTE 1: Umsatz pro Monat
    WITH monat_gesamt AS (
        SELECT monat, SUM(umsatz) AS gesamt
        FROM verkauf
        GROUP BY monat
    ),
    -- CTE 2: Durchschnitt über alle Monate
    monat_avg AS (
        SELECT AVG(gesamt) AS avg_umsatz
        FROM monat_gesamt
    )
    -- Hauptabfrage: Monate über/unter Durchschnitt
    SELECT
        m.monat,
        m.gesamt,
        ROUND(a.avg_umsatz, 0) AS durchschnitt,
        CASE
            WHEN m.gesamt >= a.avg_umsatz THEN 'ueber Durchschnitt'
            ELSE 'unter Durchschnitt'
        END AS bewertung
    FROM monat_gesamt m, monat_avg a
    ORDER BY m.monat
"""
print("Monatsbewertung mit CTE:")
display(pd.read_sql(sql_cte, con))
con.close()

---
# 4. Interaktive Dashboards mit plotly

Mit plotly Express können interaktive Dashboards in wenigen Zeilen erstellt werden.

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    df_plot = pd.DataFrame([
        {"monat": "Jan", "region": "nord", "umsatz": 12000, "bestellungen": 8},
        {"monat": "Jan", "region": "süd",  "umsatz": 9500,  "bestellungen": 6},
        {"monat": "Jan", "region": "ost",  "umsatz": 7800,  "bestellungen": 5},
        {"monat": "Feb", "region": "nord", "umsatz": 15000, "bestellungen": 10},
        {"monat": "Feb", "region": "süd",  "umsatz": 11000, "bestellungen": 7},
        {"monat": "Feb", "region": "ost",  "umsatz": 9200,  "bestellungen": 6},
        {"monat": "Mär", "region": "nord", "umsatz": 13500, "bestellungen": 9},
        {"monat": "Mär", "region": "süd",  "umsatz": 10200, "bestellungen": 7},
        {"monat": "Mär", "region": "ost",  "umsatz": 8800,  "bestellungen": 6},
    ])

    # Interaktives Multi-Diagramm-Dashboard
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Umsatz nach Region", "Bestellungen pro Monat")
    )

    # Liniendiagramm: Umsatz-Trend
    for region in df_plot["region"].unique():
        d = df_plot[df_plot["region"] == region]
        fig.add_trace(
            go.Scatter(x=d["monat"], y=d["umsatz"], name=region, mode="lines+markers"),
            row=1, col=1
        )

    # Balkendiagramm: Bestellungen
    bestellungen_sum = df_plot.groupby("monat")["bestellungen"].sum().reset_index()
    fig.add_trace(
        go.Bar(x=bestellungen_sum["monat"], y=bestellungen_sum["bestellungen"],
               name="Gesamt", showlegend=False, marker_color="#2196F3"),
        row=1, col=2
    )

    fig.update_layout(
        title_text="Verkaufs-Dashboard Q1 2024",
        height=450,
        hovermode="x unified"
    )
    fig.show()

    # Als HTML-Datei speichern (vollständig interaktiv, kein Server nötig)
    fig.write_html("output/dashboard_interaktiv.html")
    print("Dashboard gespeichert: output/dashboard_interaktiv.html")

except ImportError:
    print("plotly nicht verfügbar — Abschnitt wird übersprungen")

---
# 5. Logging in ETL-Pipelines

`print()` ist gut zum Lernen, aber in produktiven Pipelines verwendet man `logging`. Vorteile:
- Log-Level (DEBUG, INFO, WARNING, ERROR) — gefiltert je nach Umgebung
- Zeitstempel automatisch
- Ausgabe in Datei und Konsole gleichzeitig
- Standard in professionellen Python-Projekten

In [ ]:
import logging

# Logger konfigurieren
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),                          # Ausgabe in Konsole
        logging.FileHandler("output/pipeline.log"),       # Ausgabe in Datei
    ]
)

logger = logging.getLogger("etl_pipeline")

# Log-Level von niedrig nach hoch:
logger.debug("Debug: sehr detaillierte Infos (standard ausgeblendet)")
logger.info("Info: Pipeline gestartet")
logger.warning("Warning: 3 fehlende Werte gefunden")
logger.error("Error: Datei nicht gefunden (Beispiel)")

In [ ]:
# ETL-Pipeline mit Logging
def extract_mit_logging(dateipfad):
    logger.info(f"[EXTRACT] Starte: {dateipfad}")
    try:
        df = pd.read_csv(dateipfad)
        logger.info(f"[EXTRACT] Fertig: {len(df)} Zeilen, {len(df.columns)} Spalten")
        return df
    except FileNotFoundError:
        logger.error(f"[EXTRACT] Datei nicht gefunden: {dateipfad}")
        raise

def transform_mit_logging(df):
    logger.info(f"[TRANSFORM] Starte mit {len(df)} Zeilen")

    n_duplikate = df.duplicated().sum()
    if n_duplikate > 0:
        logger.warning(f"[TRANSFORM] {n_duplikate} Duplikate gefunden und entfernt")
        df = df.drop_duplicates()

    n_fehlend = df.isnull().sum().sum()
    if n_fehlend > 0:
        logger.warning(f"[TRANSFORM] {n_fehlend} fehlende Werte gefunden")

    logger.info(f"[TRANSFORM] Fertig: {len(df)} Zeilen")
    return df

# Test
df_test = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "wert": [10, 20, 20, None]
})
# Speichere als CSV zum Testen
df_test.to_csv("daten/test_log.csv", index=False)

df_roh = extract_mit_logging("daten/test_log.csv")
df_fertig = transform_mit_logging(df_roh)

In [ ]:
# Log-Datei anzeigen
with open("output/pipeline.log") as f:
    print(f.read())

---
# 6. Große Dateien mit `chunksize` verarbeiten

Wenn eine CSV-Datei nicht in den Arbeitsspeicher passt, kann pandas sie in **Chunks** (Abschnitte) stückweise verarbeiten.

Typisch ab ca. 1 GB CSV-Dateigröße relevant.

In [ ]:
# Simuliere eine größere CSV-Datei
n_zeilen = 100_000
df_gross = pd.DataFrame({
    "id":       range(n_zeilen),
    "region":   np.random.choice(["nord", "süd", "ost"], n_zeilen),
    "kategorie":np.random.choice(["Elektronik", "Zubehör", "Speicher"], n_zeilen),
    "umsatz":   np.random.uniform(10, 1000, n_zeilen).round(2),
})
df_gross.to_csv("daten/grosse_datei.csv", index=False)
print(f"Testdatei erstellt: {os.path.getsize('daten/grosse_datei.csv') / 1024:.0f} KB")

In [ ]:
# Chunk-basierte Verarbeitung
# Statt die gesamte Datei zu laden, lesen wir 10.000 Zeilen auf einmal

chunk_groesse = 10_000
umsatz_gesamt_pro_region = {}  # Akkumulator
zeilen_gesamt = 0

# pd.read_csv mit chunksize gibt einen Iterator zurück
for chunk in pd.read_csv("daten/grosse_datei.csv", chunksize=chunk_groesse):
    zeilen_gesamt += len(chunk)

    # Aggregation auf dem Chunk
    chunk_agg = chunk.groupby("region")["umsatz"].sum()

    # Zum Gesamtakkumulator hinzufügen
    for region, summe in chunk_agg.items():
        umsatz_gesamt_pro_region[region] = umsatz_gesamt_pro_region.get(region, 0) + summe

print(f"Verarbeitete Zeilen: {zeilen_gesamt:,}")
print("Gesamtumsatz pro Region:")
for region, summe in sorted(umsatz_gesamt_pro_region.items()):
    print(f"  {region}: {summe:,.0f} Euro")

In [ ]:
# Eleganter: Chunks filtern und nur relevante Daten behalten
# Beispiel: Nur Elektronik-Bestellungen über 500 Euro extrahieren

gefilterte_chunks = []

for chunk in pd.read_csv("daten/grosse_datei.csv", chunksize=chunk_groesse):
    relevante_zeilen = chunk[
        (chunk["kategorie"] == "Elektronik") & (chunk["umsatz"] > 500)
    ]
    if len(relevante_zeilen) > 0:
        gefilterte_chunks.append(relevante_zeilen)

df_gefiltert = pd.concat(gefilterte_chunks, ignore_index=True)
print(f"Gefilterte Zeilen: {len(df_gefiltert):,} von {n_zeilen:,} ({100*len(df_gefiltert)/n_zeilen:.1f}%)")
df_gefiltert.head()

### Übung 2: Chunk-Verarbeitung

Berechne aus der großen CSV mit Chunk-Verarbeitung:
1. Den Durchschnittsumsatz pro Kategorie
2. Die Anzahl der Bestellungen pro Kategorie

Tipp: Du musst sowohl Summe als auch Anzahl akkumulieren, um am Ende den Durchschnitt berechnen zu können (`avg = summe / anzahl`).

In [ ]:
# LÖSUNG
summen = {}
anzahlen = {}

for chunk in pd.read_csv("daten/grosse_datei.csv", chunksize=10_000):
    for kat, gruppe in chunk.groupby("kategorie"):
        summen[kat]   = summen.get(kat, 0)   + gruppe["umsatz"].sum()
        anzahlen[kat] = anzahlen.get(kat, 0) + len(gruppe)

print(f"{'Kategorie':<15} {'Anzahl':>8} {'Ø Umsatz':>12}")
print("-" * 38)
for kat in sorted(summen):
    avg = summen[kat] / anzahlen[kat]
    print(f"{kat:<15} {anzahlen[kat]:>8,} {avg:>11.2f} €")

---
# Zusammenfassung

| Thema | Wann einsetzen |
|-------|----------------|
| **Regex** | Unstrukturierte Texte bereinigen, Werte extrahieren |
| **Parquet** | Datenpipelines mit großen oder wiederholt gelesenen Dateien |
| **Window Functions** | Rang, laufende Summen, Vergleich mit Gruppenwert in SQL |
| **CTEs** | Lange SQL-Abfragen lesbarer strukturieren |
| **plotly** | Interaktive Reports und Dashboards für Stakeholder |
| **logging** | Produktive Pipelines, Fehlerdiagnose, Audit-Trails |
| **chunksize** | CSV-Dateien größer als verfügbarer RAM |

Diese Techniken sind der nächste Schritt hin zu professionellen, produktionstauglichen Datenpipelines.

---
# Musterlösungen — Weitere Übungsaufgaben

### Aufgabe 1 — Regex Datenextraktion

Extrahiere aus der Spalte `rohdaten` folgende Informationen mit Regex:
- `artikel_nr` — Format `ART-XXXX` (4 Ziffern)
- `preis` — Zahl nach `'Preis '` (als float)
- `mwst_satz` — Zahl vor `'%'` (als int)

In [ ]:
# LÖSUNG
import pandas as pd

df_roh = pd.DataFrame({'rohdaten': [
    'Artikel ART-1042: Laptop Pro, Preis 999 Euro, MwSt 19%',
    'Artikel ART-0815: USB-Maus, Preis 29.90 Euro, MwSt 19%',
    'Artikel ART-2301: Fachbuch Python, Preis 39 Euro, MwSt 7%',
    'Artikel ART-9999: Monitor 27Zoll, Preis 349 Euro, MwSt 19%',
]})

df_roh['artikel_nr'] = df_roh['rohdaten'].str.extract(r'(ART-\d{4})')
df_roh['preis']      = df_roh['rohdaten'].str.extract(r'Preis ([\d.]+)').astype(float)
df_roh['mwst_satz']  = df_roh['rohdaten'].str.extract(r'(\d+)%').astype(int)
display(df_roh[['artikel_nr','preis','mwst_satz']])


### Aufgabe 2 — SQL Window Function

Schreibe eine SQL-Abfrage, die pro Monat folgendes berechnet:
1. Den Umsatz
2. Den kumulativen Umsatz bis zu diesem Monat
3. Die prozentuale Veränderung zum Vormonat

Tipp: `LAG(umsatz) OVER (ORDER BY monat)` liefert den Wert des Vormonats.

In [ ]:
# LÖSUNG
import pandas as pd, sqlite3

con = sqlite3.connect(':memory:')
pd.DataFrame({'monat': range(1,7),
              'umsatz': [41000,38000,45000,52000,49000,61000]})\
  .to_sql('umsatz', con, index=False)

sql = """
    SELECT
        monat,
        umsatz,
        SUM(umsatz) OVER (ORDER BY monat) AS kumulativ,
        ROUND(
            100.0 * (umsatz - LAG(umsatz) OVER (ORDER BY monat))
                  / LAG(umsatz) OVER (ORDER BY monat), 1
        ) AS veraenderung_pct
    FROM umsatz
    ORDER BY monat
"""
display(pd.read_sql(sql, con))
con.close()
